# 06 · Evaluación comparativa de modelos de recuperación

Genera la tabla final que se reportará en la exposición.

- Ground truth construido por **subject**: documentos del mismo `subject` que la consulta son relevantes.
- Consultas extraídas de `data/queries/eval_queries.csv` (o muestreo automático).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.retrieval import TfidfRetriever, BM25Retriever, SemanticRetriever
from src.evaluation import evaluate_retriever

df = pd.read_parquet(config.CORPUS_PATH).head(5000).reset_index(drop=True)
df.head(2)

In [ ]:
# Construir consultas + ground truth a partir de subject
rng = np.random.default_rng(config.SEED)
queries, ground_truth = [], []
for subj, group in df.groupby('subject'):
    if len(group) < 10:
        continue
    sample = group.sample(min(3, len(group)), random_state=config.SEED)
    for _, row in sample.iterrows():
        queries.append(row['title'])
        ground_truth.append(group.index.tolist())
print('Queries:', len(queries))

In [ ]:
tfidf = TfidfRetriever().fit(df['clean_text'].tolist())
bm25 = BM25Retriever().fit(df['clean_text'].tolist())
sem = SemanticRetriever().fit(df['text'].tolist())

rows = [evaluate_retriever(r, queries, ground_truth) for r in (tfidf, bm25, sem)]
results_df = pd.DataFrame(rows).set_index('name').round(4)
results_df

In [ ]:
results_df.to_csv(config.REPORTS_DIR / 'comparativa.csv')

fig, ax = plt.subplots(figsize=(7, 4))
results_df[['P@5','R@5','NDCG@5','MAP']].plot.bar(ax=ax)
ax.set_title('Comparación de modelos de recuperación')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'comparativa.png', dpi=120)
plt.show()